The original [CountBench](https://arxiv.org/abs/2302.12066) is a dataset that contains 540 images with 2-10 objects and captions describing the image as well as the number of objects to count. However, the original CountBench captions includes the number of objects in the caption and includes images with links that are unavailable. The authors of [CountGD](https://arxiv.org/pdf/2407.04619) released a cleaned version of CountBench, a 504-image
subset of CountBench that resolves these issues. We want to use this dataset.

However, the CountGD authors left the dataset in a scattered state. To obtain the cleaned CountGD dataset, follow these [instructions](https://github.com/niki-amini-naieni/CountGD/issues/6#issuecomment-2573026161):
1. Download and unzip the directory of images from google drive: https://drive.google.com/file/d/1kGUzF8C-z7HOBvMWjLhrUQtvY5bx6fQ9/view
2. Download the metadata for the images from here: https://github.com/niki-amini-naieni/CountGD/blob/main/data/fsc147_gdino/CountBench.json

Make sure the `CountBench` directory with all the images and the metadata are in the same directory. Now run the following code:

In [1]:
!pip install datasets huggingface_hub

In [2]:
import os, json
from textwrap import dedent
from datasets import Dataset, Image, DatasetDict

/Users/eitanturok/Visual-RFT/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Add metadata to the dataset
metadata_path = "CountBench.json"
with open(metadata_path, 'r') as file: metadata = json.load(file)
dataset1 = [{"id": int(id_.strip('.jpg'))} | data for id_, data in metadata.items()]

dataset1[:3]

[{'id': 1,
  'image_url': 'http://www.wirerealm.com/wp-content/uploads/2015/08/top-10-best-gaming-headsets-1024x448.jpg',
  'count': 10,
  'text': 'the best gaming headsets in the market',
  'keyphrases': ['headsets']},
 {'id': 10,
  'image_url': 'https://cdn3.volusion.com/lkzsf.javck/v/vspfiles/photos/HTH11-2T.jpg?1485341377',
  'count': 9,
  'text': 'the bird and birdhouse patterns, wrapped on blocks instead of joined to make a full quilt',
  'keyphrases': ['patterns']},
 {'id': 100,
  'image_url': 'https://i.imgur.com/tKLJJ.jpg',
  'count': 5,
  'text': 'the holiday nail polishes',
  'keyphrases': ['nail polishes']}]

In [4]:
# Add images to the dataset
image_dir = "CountBench"
id_to_image_paths = {int(image_path.strip('.jpg')): os.path.join(image_dir, image_path) for image_path in os.listdir(image_dir)}
dataset2 = [{'image': id_to_image_paths[data['id']]} | data for data in dataset1]

dataset2[:3]

[{'image': 'CountBench/1.jpg',
  'id': 1,
  'image_url': 'http://www.wirerealm.com/wp-content/uploads/2015/08/top-10-best-gaming-headsets-1024x448.jpg',
  'count': 10,
  'text': 'the best gaming headsets in the market',
  'keyphrases': ['headsets']},
 {'image': 'CountBench/10.jpg',
  'id': 10,
  'image_url': 'https://cdn3.volusion.com/lkzsf.javck/v/vspfiles/photos/HTH11-2T.jpg?1485341377',
  'count': 9,
  'text': 'the bird and birdhouse patterns, wrapped on blocks instead of joined to make a full quilt',
  'keyphrases': ['patterns']},
 {'image': 'CountBench/100.jpg',
  'id': 100,
  'image_url': 'https://i.imgur.com/tKLJJ.jpg',
  'count': 5,
  'text': 'the holiday nail polishes',
  'keyphrases': ['nail polishes']}]

In [5]:
# Convert to Hugging Face Dataset
# Cast the "image" column to the Image type
dataset3 = Dataset.from_list(dataset2).cast_column("image", Image())

dataset3[:3]

{'image': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=877x384>,
  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=389x384>,
  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=576x384>],
 'id': [1, 10, 100],
 'image_url': ['http://www.wirerealm.com/wp-content/uploads/2015/08/top-10-best-gaming-headsets-1024x448.jpg',
  'https://cdn3.volusion.com/lkzsf.javck/v/vspfiles/photos/HTH11-2T.jpg?1485341377',
  'https://i.imgur.com/tKLJJ.jpg'],
 'count': [10, 9, 5],
 'text': ['the best gaming headsets in the market',
  'the bird and birdhouse patterns, wrapped on blocks instead of joined to make a full quilt',
  'the holiday nail polishes'],
 'keyphrases': [['headsets'], ['patterns'], ['nail polishes']]}

In [6]:
# Make solution column
def solution_col(example):
    # Create the new 'solution' column with the desired format
    example["solution"] = f"<answer>{example['count']}</answer>"
    return example

dataset4 = dataset3.map(solution_col)
dataset4[:3]

Map: 100%|██████████| 504/504 [00:00<00:00, 26498.31 examples/s]


{'image': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=877x384>,
  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=389x384>,
  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=576x384>],
 'id': [1, 10, 100],
 'image_url': ['http://www.wirerealm.com/wp-content/uploads/2015/08/top-10-best-gaming-headsets-1024x448.jpg',
  'https://cdn3.volusion.com/lkzsf.javck/v/vspfiles/photos/HTH11-2T.jpg?1485341377',
  'https://i.imgur.com/tKLJJ.jpg'],
 'count': [10, 9, 5],
 'text': ['the best gaming headsets in the market',
  'the bird and birdhouse patterns, wrapped on blocks instead of joined to make a full quilt',
  'the holiday nail polishes'],
 'keyphrases': [['headsets'], ['patterns'], ['nail polishes']],
 'solution': ['<answer>10</answer>',
  '<answer>9</answer>',
  '<answer>5</answer>']}

In [7]:
# Add a problem column
def problem_col(example):
    if len(example['keyphrases']) == 2:
        objects = ' of '.join(example['keyphrases'])
    elif len(example['keyphrases']) == 3:
        objects = f"{example['keyphrases'][0]} of {example['keyphrases'][1]} and {example['keyphrases'][2]}"
    else:
        objects = example['keyphrases'][0]

    example['problem'] = dedent(f"""Count the number of {objects} in this image. Output the thinking process in <think> </think> and final answer in <answer> </answer> tags.The output answer format should be as follows:
    <think> ... </think> <answer>NUMBER</answer>
    NUMBER must be an integer consisting of digits, e.g. 11, not eleven.
    Please strictly follow the format.""")
    return example

dataset5 = dataset4.map(problem_col)
dataset5[:3]

Map: 100%|██████████| 504/504 [00:00<00:00, 22371.99 examples/s]


{'image': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=877x384>,
  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=389x384>,
  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=576x384>],
 'id': [1, 10, 100],
 'image_url': ['http://www.wirerealm.com/wp-content/uploads/2015/08/top-10-best-gaming-headsets-1024x448.jpg',
  'https://cdn3.volusion.com/lkzsf.javck/v/vspfiles/photos/HTH11-2T.jpg?1485341377',
  'https://i.imgur.com/tKLJJ.jpg'],
 'count': [10, 9, 5],
 'text': ['the best gaming headsets in the market',
  'the bird and birdhouse patterns, wrapped on blocks instead of joined to make a full quilt',
  'the holiday nail polishes'],
 'keyphrases': [['headsets'], ['patterns'], ['nail polishes']],
 'solution': ['<answer>10</answer>',
  '<answer>9</answer>',
  '<answer>5</answer>'],
 'problem': ['Count the number of headsets in this image. Output the thinking process in <think> </think> and final answer in <answer> </answer> tags.The output answer format should be 

In [8]:
# train, test split
dataset6 = dataset5.train_test_split(test_size=0.2, shuffle=True, seed=42)

dataset6

DatasetDict({
    train: Dataset({
        features: ['image', 'id', 'image_url', 'count', 'text', 'keyphrases', 'solution', 'problem'],
        num_rows: 403
    })
    test: Dataset({
        features: ['image', 'id', 'image_url', 'count', 'text', 'keyphrases', 'solution', 'problem'],
        num_rows: 101
    })
})

In [9]:
# Upload dataset to hf
repo_id = "eturok/count-bench-clean"
dataset = dataset6

# Push the dataset to the Hub
dataset.push_to_hub(repo_id, token=os.getenv('HF_TOKEN'))

Uploading the dataset shards: 100%|██████████| 1/1 [00:06<00:00,  6.11s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/eturok/count-bench-clean/commit/41bbd30655e15d56bc2060a3025c2749cdbf048f', commit_message='Upload dataset', commit_description='', oid='41bbd30655e15d56bc2060a3025c2749cdbf048f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/eturok/count-bench-clean', endpoint='https://huggingface.co', repo_type='dataset', repo_id='eturok/count-bench-clean'), pr_revision=None, pr_num=None)